# mse-reconstruction-loss — faded example 1: Compute MSE only over valid (non-masked) pixels

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `mse-reconstruction-loss`. Running the beacon reports progress on the `Generative: MSE reconstruction loss` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Generative: MSE reconstruction loss` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`mse-reconstruction-loss`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "mse-reconstruction-loss"
DD_SUBTOPIC = "Generative: MSE reconstruction loss"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

When an image batch contains masked regions (e.g., padding or missing data), you often want to compute the reconstruction loss only over valid pixels. The standard approach is to compute per-element MSE with `reduction='none'`, zero out the invalid positions using a binary mask, then divide the total by the number of valid pixels to get a proper mean.

## Faded exercise 1

Implement `masked_mse(pred, target, mask)` where:
- `pred`, `target`: `(B, C, H, W)` float tensors
- `mask`: `(B, 1, H, W)` binary float tensor (1.0 = valid, 0.0 = masked out)

Algorithm:
1. Compute per-element MSE: `per_elem = F.mse_loss(pred, target, reduction='none')` → `(B, C, H, W)`.
2. Zero out invalid positions: multiply by `mask` (which broadcasts over the C dimension).
3. Divide the total by the number of valid elements: `mask.sum() * C` (where `C = pred.shape[1]`).
4. Return the scalar.

Your task: **fill in steps 2–4**: the masked sum, the valid-element count, and the division.

**Fill in:** Zeroing masked pixels with element-wise multiplication, computing the count of valid elements (mask.sum() * C), and returning the total divided by that count.

In [ ]:
import torch
import torch.nn.functional as F

def masked_mse(pred: torch.Tensor, target: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    C = pred.shape[1]
    per_elem = F.mse_loss(pred, target, reduction='none')  # (B, C, H, W)
    masked   = per_elem * mask                             # zero out invalid pixels
    n_valid  = mask.sum() * C                              # total valid element count
    return masked.sum() / n_valid

def _test():
    import torch, torch.nn.functional as F
    torch.manual_seed(0)
    B, C, H, W = 2, 3, 4, 4
    pred   = torch.randn(B, C, H, W)
    target = torch.randn(B, C, H, W)
    mask_all = torch.ones(B, 1, H, W)
    loss_all = masked_mse(pred, target, mask_all)
    loss_plain = F.mse_loss(pred, target)
    assert abs(loss_all.item() - loss_plain.item()) < 1e-5
    mask_half = torch.zeros(B, 1, H, W)
    mask_half[:, :, :H//2, :] = 1.0
    loss_half = masked_mse(pred, target, mask_half)
    assert loss_half.item() > 0


def _test():
    import torch, torch.nn.functional as F
    torch.manual_seed(0)
    B, C, H, W = 2, 3, 4, 4
    pred   = torch.randn(B, C, H, W)
    target = torch.randn(B, C, H, W)
    # All-ones mask should match plain MSE
    mask_all = torch.ones(B, 1, H, W)
    loss_full = masked_mse(pred, target, mask_all)
    loss_ref  = F.mse_loss(pred, target)
    assert abs(loss_full.item() - loss_ref.item()) < 1e-5, f"full mask mismatch: {loss_full.item()} vs {loss_ref.item()}"
    # All-zeros mask: numerator is 0 but denominator also 0 -> we just check no crash and result is 0 or nan
    # Partial mask: result > 0 and != plain MSE
    mask_half = torch.zeros(B, 1, H, W)
    mask_half[:, :, :H//2, :] = 1.0
    loss_half = masked_mse(pred, target, mask_half)
    assert loss_half.item() > 0, "partial mask loss should be positive"
    assert abs(loss_half.item() - loss_ref.item()) > 1e-4, "partial mask should differ from plain MSE"


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch
import torch.nn.functional as F

def masked_mse(pred: torch.Tensor, target: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    C = pred.shape[1]
    per_elem = F.mse_loss(pred, target, reduction='none')  # (B, C, H, W)
    masked   = per_elem * mask                             # zero out invalid pixels
    n_valid  = mask.sum() * C                              # total valid element count
    return masked.sum() / n_valid

def _test():
    import torch, torch.nn.functional as F
    torch.manual_seed(0)
    B, C, H, W = 2, 3, 4, 4
    pred   = torch.randn(B, C, H, W)
    target = torch.randn(B, C, H, W)
    mask_all = torch.ones(B, 1, H, W)
    loss_all = masked_mse(pred, target, mask_all)
    loss_plain = F.mse_loss(pred, target)
    assert abs(loss_all.item() - loss_plain.item()) < 1e-5
    mask_half = torch.zeros(B, 1, H, W)
    mask_half[:, :, :H//2, :] = 1.0
    loss_half = masked_mse(pred, target, mask_half)
    assert loss_half.item() > 0
```
</details>